# 02 - Build `ml.bus_matching_avl_positions`

Second notebook for the Bus Matching model: a view over
`silver.avl_pings`, scoped to the same `[2023-10-31, 2023-12-02)` window
as `01_build_candidate_pairs.ipynb` ("November 2023 +/- one day"),
renaming/reformatting a few columns for this model's use.

## Why a view, not a materialized table

Originally built as a full copy: confirmed live that the window is
~140M rows (the November 2023 partition alone is 130,957,352 rows /
39 GB), and that filtering to only the devices in
`ml.bus_matching_candidate_pairs` wouldn't meaningfully shrink that
(1,402 of 1,493 distinct devices pinging in November are already in the
candidate table -- row count is driven by ping frequency, not device
count). Stepping back: the only actual value-add over querying
`silver.avl_pings` directly is a few renamed/reformatted columns (below)
-- nothing that requires duplicating the data. A view gets the same
renaming/formatting for free: it's a query rewrite, not a copy, so it
costs no storage and reads through it use exactly the indexes
`silver.avl_pings` already has (`ts_idx` everywhere, plus
`device_ts_idx`/`vehicle_ts_idx` on the November partition). The one
thing a view can't offer is a real surrogate primary key -- not needed
here, since the natural use of this data is "positions for a given
device in a given window," not "look up one specific position by id."

## Column choices, and why they differ from `silver.avl_pings`

- `device_id`, `metric_timestamp`, `latitude`, `longitude`, `speed`,
  `geom`: passed through unchanged -- every existing `ml.trip_validity_*`
  table that carries AVL columns forward keeps these names identical
  rather than inventing new ones.
- `heading_degrees` (was `direction`): renamed. It's the GPS unit's
  compass heading, not a route direction -- colliding in name (though
  not meaning) with `route_direction`/`direction` elsewhere in
  `ml.trip_validity_*`, which are AFC's binary out/back flag. A small
  sample suggested a clean 0-359 range, but the full window tells a
  different story: confirmed live that it actually runs 0-9360 in this
  window, on exactly 10 of the 140,655,482 rows (9 at 360, 1 at 9360 --
  both, as it happens, exact multiples of 360). Rather than pass those
  10 through as nonsensical headings, they're mapped to 0 in the view
  itself (`direction >= 360` -> `0`).
- `route_id` (was `route_code`, integer): cast to text and zero-padded
  to 3 characters (only when shorter -- never truncating), exactly
  matching how `ml.trip_validity_trips.route_id` formats AFC's own
  `line_number`. Same column name too, for consistency with that table.


In [1]:
import os
from pathlib import Path

import psycopg
from psycopg import sql

In [2]:
_root = Path.cwd()
while not (_root / "pyproject.toml").exists():
    _root = _root.parent
os.chdir(_root)
os.environ.setdefault("RAW_DATA_ROOT", str(_root))

'/home/victor/repos/opa-database'

In [3]:
from opa_database.config import settings

WINDOW_START = "2023-10-31"
WINDOW_END = "2023-12-02"

conn = psycopg.connect(settings.db_dsn)
conn.execute("CREATE SCHEMA IF NOT EXISTS ml;")
conn.commit()
print("ml schema ready")

ml schema ready


## `ml.bus_matching_avl_positions`

The window bounds are baked into the view definition (a view takes no
runtime parameters), built from the same `WINDOW_START`/`WINDOW_END`
constants as notebook 01 rather than hand-typed a second time.

In [4]:
conn.execute(
    sql.SQL(
        """
        CREATE OR REPLACE VIEW ml.bus_matching_avl_positions AS
        SELECT
            device_id,
            metric_timestamp,
            latitude,
            longitude,
            CASE WHEN direction >= 360 THEN 0 ELSE direction END
                AS heading_degrees,
            speed,
            CASE WHEN length(route_code::text) < 3
                 THEN lpad(route_code::text, 3, '0')
                 ELSE route_code::text
            END AS route_id,
            geom
        FROM silver.avl_pings
        WHERE metric_timestamp >= {start} AND metric_timestamp < {end};
        """
    ).format(start=sql.Literal(WINDOW_START), end=sql.Literal(WINDOW_END))
)
conn.commit()
print("ml.bus_matching_avl_positions view created")

ml.bus_matching_avl_positions view created


## Sanity check

In [5]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT
            count(*) AS rows,
            count(DISTINCT device_id) AS distinct_devices,
            count(DISTINCT route_id) AS distinct_routes,
            min(metric_timestamp) AS earliest,
            max(metric_timestamp) AS latest
        FROM ml.bus_matching_avl_positions;
    """)
    print(cur.fetchone())

    cur.execute("""
        SELECT min(heading_degrees), max(heading_degrees), min(speed), max(speed)
        FROM ml.bus_matching_avl_positions;
    """)
    print("heading_degrees min/max, speed min/max:", cur.fetchone())

    cur.execute("SELECT route_id FROM ml.bus_matching_avl_positions LIMIT 5;")
    print("sample route_id:", [r[0] for r in cur.fetchall()])

(140655482, 1495, 348, datetime.datetime(2023, 10, 31, 0, 0, tzinfo=zoneinfo.ZoneInfo(key='Etc/UTC')), datetime.datetime(2023, 12, 1, 23, 59, 59, tzinfo=zoneinfo.ZoneInfo(key='Etc/UTC')))


heading_degrees min/max, speed min/max: (0, 359, 0, 199)
sample route_id: ['000', '000', '395', '000', '000']
